In [ ]:
from ElasticSketch import ElasticSketch
import math
import time
import pandas as pd
import scipy as sp
import numpy as np
from tqdm import tqdm

In [ ]:
path = '/path/to/appraise.csv'
df = pd.read_csv(path)
df

,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS
0,3.257485e+09,20828.0,3.260381e+09,52.0,18.0,179430.0,2955.0,272656.0,2576.0
1,3.253771e+09,19149.0,3.249148e+09,466.0,17.0,26506.0,3318.0,2749168.0,2930.0
2,2.895684e+09,52561.0,2.916716e+09,276.0,7.0,1205313.0,3052.0,792420.0,1013.0
3,3.232337e+09,62676.0,3.241048e+09,174.0,6.0,2995190.0,1983.0,2458753.0,2300.0
4,2.899580e+09,62599.0,2.909405e+09,439.0,5.0,149998.0,389.0,280651.0,2348.0
...,...,...,...,...,...,...,...,...,...
75987971,3.240715e+09,46722.0,3.236086e+09,50072.0,7.0,594195.0,1498.0,267200.0,2088.0
75987972,3.253769e+09,49278.0,3.243913e+09,1315.0,7.0,984634.0,2500.0,1443692.0,3566.0
75987973,3.236313e+09,55482.0,3.224522e+09,407.0,7.0,788061.0,605.0,1188864.0,3345.0
75987974,3.232314e+09,41090.0,3.224105e+09,598.0,6.0,2522048.0,3074.0,2310995.0,1320.0


In [3]:
# Unnecessary computation.
def faster_cdf(sketch):
    density = sketch.pmf()
    total = sum(density)
    cdf = []
    temp = 0
    for d in density:
        temp += d
        cdf.append(1.0*temp/total)
    return cdf

In [5]:
# Entropy calculation may be wrong in original...
def entropy_fix(sketch):
		density = sketch.pmf()
		H = 0
		m = sum(density)
		for i, n_i in enumerate(density):
			if i <= 0 or n_i <= 0:
				pass
			else:
				H += 1.0 * (n_i / m) * math.log( 1.0 * (n_i / m))
		return -1*H

# Elastic Evaluation

In [7]:
import statistics
import os
import json
from copy import deepcopy
from tqdm.contrib.concurrent import process_map
import concurrent

In [8]:
SAVE_FOLDER = 'eval/elastic/'
NAME = 'nfiot_dp'
os.makedirs(SAVE_FOLDER, exist_ok=True)

In [ ]:
elastic_sketches = {}

tqdm.pandas()

def compute_sketch(idx, data):
    sketch = ElasticSketch(3, 195)
    for _, item in data.items():
        sketch.insert(item)
    return sketch

print(f'{len(df.columns)} Features...')

job_args = [(i, col) for i, col in enumerate(df.columns)]
with tqdm(total=len(job_args)) as pbar:
    # let's give it some more threads:
    with concurrent.futures.ProcessPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(compute_sketch, i, df[col]): col for i, col in job_args}
        results = {}
        for future in concurrent.futures.as_completed(futures):
            col = futures[future]
            results[col] = future.result()
            pbar.update(1)
            
elastic_sketches = results
elastic_sketches

9 Features...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [09:32<00:00, 63.65s/it]


{'PROTOCOL': <ElasticSketch.ElasticSketch at 0x7f4e016f2d50>,
 'L4_DST_PORT': <ElasticSketch.ElasticSketch at 0x7f4e016f2f00>,
 'IN_PKTS': <ElasticSketch.ElasticSketch at 0x7f4e016f2db0>,
 'L4_SRC_PORT': <ElasticSketch.ElasticSketch at 0x7f4e015e2ae0>,
 'OUT_PKTS': <ElasticSketch.ElasticSketch at 0x7f4e016f3410>,
 'IN_BYTES': <ElasticSketch.ElasticSketch at 0x7f4e015e2ff0>,
 'OUT_BYTES': <ElasticSketch.ElasticSketch at 0x7f4e016f0e90>,
 'IPV4_SRC_ADDR': <ElasticSketch.ElasticSketch at 0x7f4e016f3800>,
 'IPV4_DST_ADDR': <ElasticSketch.ElasticSketch at 0x7f4e016f3680>}

## Entropy

In [10]:
entropies = {}

for col in tqdm(df.columns):
    val_freqs = df[col].value_counts()
    real_entropy = sp.stats.entropy(val_freqs)
    sketch_entropy = entropy_fix(elastic_sketches[col])
    entropies[col] = abs((real_entropy - sketch_entropy) / real_entropy)

entropies, statistics.mean(entropies.values()), statistics.stdev(entropies.values())

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:40<00:00,  4.52s/it]


({'IPV4_SRC_ADDR': 0.9237436004982296,
  'L4_SRC_PORT': 0.7184867213508176,
  'IPV4_DST_ADDR': 0.9207275148844389,
  'L4_DST_PORT': 0.6065111672428851,
  'PROTOCOL': 0.10090357225030516,
  'IN_BYTES': 0.840314589147872,
  'IN_PKTS': 0.6417198239053756,
  'OUT_BYTES': 0.8140674194153819,
  'OUT_PKTS': 0.6292714612968957},
 0.6884162077769113,
 0.2516622965608168)

In [11]:
path = os.path.join(SAVE_FOLDER, f'{NAME}_entropy.json')
out = deepcopy(entropies)

with open(path, 'w') as f:
    out.update({
        'mean': statistics.mean(entropies.values()),
        'std': statistics.stdev(entropies.values())
    })
    json.dump(out, f, indent=2)

## Quantile

In [12]:
# Copying logic of Evaluation.ipynb...
def quantile_diff(q1, q2):
    i = 0
    max_q = 0
    mean_q = 0
    while i<len(q1) and i <len(q2):
        diff = abs(q1[i] - q2[i])
        max_q = max(max_q, diff)
        mean_q = mean_q + (diff - mean_q) / (i + 1)
        i+=1
    while i<len(q1):
        diff = abs(q1[i] - q2[-1])
        max_q = max(max_q, diff)
        mean_q = mean_q + (diff - mean_q) / (i + 1)
        i+=1
    while i<len(q2):
        diff = abs(q1[-1] - q2[i])
        max_q = max(max_q, diff)
        mean_q = mean_q + (diff - mean_q) / (i + 1)
        i+=1
    return max_q, mean_q

In [13]:
# Copying logic of Evaluation.ipynb (cdf and density count frequencies?)...
real_cdfs = {}
for col in tqdm(df.columns):
    val_counts = df[col].value_counts().values
    raw_cdf_counts = np.unique(val_counts, return_counts=True)[1].cumsum()
    real_cdf = raw_cdf_counts / raw_cdf_counts[-1]
    real_cdfs[col] = real_cdf
real_cdfs

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:38<00:00,  4.32s/it]


{'IPV4_SRC_ADDR': array([0.66362983, 0.88110016, 0.9652034 , 0.9915684 , 0.99826131,
        0.999687  , 0.99995156, 0.99999325, 0.99999917, 0.9999999 ,
        1.        ]),
 'L4_SRC_PORT': array([1.52587891e-05, 3.05175781e-05, 4.57763672e-05, ...,
        9.99954224e-01, 9.99984741e-01, 1.00000000e+00]),
 'IPV4_DST_ADDR': array([0.6495933 , 0.88186464, 0.96663591, 0.99211058, 0.99840873,
        0.99972459, 0.99995753, 0.99999381, 0.99999908, 0.99999996,
        0.99999998, 1.        ]),
 'L4_DST_PORT': array([1.52587891e-05, 3.05175781e-05, 4.57763672e-05, ...,
        9.99969482e-01, 9.99984741e-01, 1.00000000e+00]),
 'PROTOCOL': array([0.00390625, 0.0078125 , 0.01171875, 0.01953125, 0.0234375 ,
        0.03125   , 0.03515625, 0.0390625 , 0.04296875, 0.046875  ,
        0.0546875 , 0.0625    , 0.06640625, 0.0703125 , 0.07421875,
        0.078125  , 0.0859375 , 0.09375   , 0.09765625, 0.109375  ,
        0.11328125, 0.12109375, 0.13671875, 0.1484375 , 0.15625   ,
        0.171875  

In [14]:
quantiles = {}

for col in tqdm(df.columns):
    diff = quantile_diff(real_cdfs[col], faster_cdf(elastic_sketches[col]))
    quantiles[col] = diff

quantiles

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:08<00:00,  1.12it/s]


{'IPV4_SRC_ADDR': (0.6636298349672516, 0.04724180599593646),
 'L4_SRC_PORT': (0.5873162841796875, 0.3435443776242641),
 'IPV4_DST_ADDR': (0.6495932970910697, 0.04677698014535053),
 'L4_DST_PORT': (0.84, 0.8338934198631084),
 'PROTOCOL': (0.7344044816080696, 0.05199566026117518),
 'IN_BYTES': (0.3466509975195962, 0.05580649580839448),
 'IN_PKTS': (0.84, 0.8249064464061756),
 'OUT_BYTES': (0.3642822011319572, 0.06182425931944042),
 'OUT_PKTS': (0.856156501726122, 0.8272707820866171)}

In [15]:
max_quantiles = [q[0] for q in quantiles.values()]
mean_quantiles = [q[1] for q in quantiles.values()]
max_quantile_stats = ( statistics.mean(max_quantiles), statistics.stdev(max_quantiles) )
mean_quantile_stats = ( statistics.mean(mean_quantiles), statistics.stdev(mean_quantiles) )



print(f'Max Quantile Stats :{max_quantile_stats}')
print(f'Mean Quantile Stats :{mean_quantile_stats}')

Max Quantile Stats :(0.6535592886915282, 0.19348160985897356)
Mean Quantile Stats :(0.3436955808344958, 0.37569414480806884)


In [125]:
for col in df.columns:
    print(col, len(real_cdfs[col]), len(faster_cdf(elastic_sketches[col])))

IPV4_SRC_ADDR 302 3701352
IPV4_DST_ADDR 831 3179374
IN_PKTS 970 5171631
IN_BYTES 1372 2859434
OUT_PKTS 796 8461408
OUT_BYTES 986 8461408
L4_SRC_PORT 844 3479786
L4_DST_PORT 544 3480717
PROTOCOL 6 8171586
TOTAL_FLOWS_EXP 6 2501
ANOMALY 2 10805785


In [16]:
path = os.path.join(SAVE_FOLDER, f'{NAME}_quantiles.json')
out = {}
for key, vals in quantiles.items():
    entry = {}
    max_q = vals[0]
    mean_q = vals[1]
    entry['max_quantile_diff'] = max_q
    entry['mean_quantile_diff'] = mean_q
    out[key] = entry
means = {
    'max_quantile_diff': max_quantile_stats[0],
    'mean_quantile_diff': mean_quantile_stats[0]
}
stds = {
    'max_quantile_diff': max_quantile_stats[1],
    'mean_quantile_diff': mean_quantile_stats[1]
}
out['mean'] = means
out['std'] = stds

with open(path, 'w') as f:
    json.dump(out, f, indent=2)